# VHealthQA Preprocessing

Xem xét dữ liệu VHealthQA sau khi tiền xử lý từ CSV sang JSONL

In [ ]:
import json
import os
from pathlib import Path

# Auto-detect project root
current_file = Path(globals().get('__file__', '.')).resolve()
if current_file.name == 'preprocessing_vhealthqa.ipynb':
    notebook_dir = current_file.parent
else:
    # Running from notebook without __file__
    notebook_dir = Path.cwd()

# Find project root (contains main.py)
project_root = notebook_dir
while project_root != project_root.parent and not (project_root / 'main.py').exists():
    project_root = project_root.parent

if not (project_root / 'main.py').exists():
    project_root = Path('d:/reranking_rag_and_qw')

os.chdir(project_root)
print(f'Project root: {project_root}')

## 1. Check processed data

In [ ]:
# Check file sizes
processed_dir = project_root / 'data' / 'vhealthqa' / 'processed'

for split in ['train', 'validation', 'test']:
    jsonl_file = processed_dir / f'{split}.jsonl'
    if jsonl_file.exists():
        with open(jsonl_file, 'r', encoding='utf-8') as f:
            count = sum(1 for _ in f)
        size_mb = jsonl_file.stat().st_size / (1024**2)
        print(f'{split:12} | {count:6} records | {size_mb:.2f} MB')
    else:
        print(f'{split:12} | NOT FOUND')

## 2. Sample records

In [ ]:
# Load one sample from each split
for split in ['train', 'validation', 'test']:
    jsonl_file = processed_dir / f'{split}.jsonl'
    if jsonl_file.exists():
        with open(jsonl_file, 'r', encoding='utf-8') as f:
            record = json.loads(f.readline())
        
        print(f'\n🔹 {split.upper()} - Sample Record:')
        print(f'  ID: {record["id"]}')
        print(f'  Question: {record["question"][:100]}...')
        print(f'  Answer: {record["answer"][:100]}...')
        print(f'  Link: {record["link"]}')

## 3. Data statistics

In [ ]:
# Load all train records for statistics
train_jsonl = processed_dir / 'train.jsonl'
train_records = []
with open(train_jsonl, 'r', encoding='utf-8') as f:
    for line in f:
        train_records.append(json.loads(line))

# Stats
question_lengths = [len(r['question'].split()) for r in train_records]
answer_lengths = [len(r['answer'].split()) for r in train_records]

print('📊 Train Set Statistics:')
print(f'  Total records: {len(train_records)}')
print(f'  Avg question length: {sum(question_lengths)/len(question_lengths):.1f} words')
print(f'  Avg answer length: {sum(answer_lengths)/len(answer_lengths):.1f} words')
print(f'  Max answer length: {max(answer_lengths)} words')
print(f'  Min answer length: {min(answer_lengths)} words')

## 4. Baseline dataset

In [ ]:
# Check baseline
baseline_file = project_root / 'evaluation' / 'baseline_dataset_from_vhealthqa.json'
if baseline_file.exists():
    with open(baseline_file, 'r', encoding='utf-8') as f:
        baseline = json.load(f)
    
    print(f'✅ Baseline dataset created')
    print(f'  Name: {baseline["name"]}')
    print(f'  Description: {baseline["description"]}')
    print(f'  Samples: {len(baseline["samples"])}')
else:
    print('❌ Baseline file not found')